## Import the Model

In [1]:
import sys
sys.path.append('../')

import src.TagModel as model
import src.TagModel_lat as model_latency
import src.Auth as Auth
import utils.utils as utils
import numpy as np
import matplotlib.pyplot as plt 
import json
import os
import threading

### Run the Model

Just considering the E[A]

In [ ]:
m_nrs = [20,30]
t_nrs = [5,10,15]
p = [.95, .9, .85, .8, .75, .7]
q = [1]


def run_optimize(parameters):
    os.system("python3 optimize.py \'"+json.dumps(parameters)+"\'")

threads = []
for m_nr in m_nrs:
    for t_nr in t_nrs:
        for p_ in p:
            for q_ in q:
                parameters = {'m_nr': m_nr, 't_nr': t_nr, 
                              'p': p_, 'q': q_, 
                              'TagEveryMessage': True, 
                              'AtLeastOnce': False, 
                              'EquivalentA': True}
                threads.append(threading.Thread(target=run_optimize, args=(parameters,)))
                threads[-1].start() 
        cnt = 0
        for thread in threads:
            thread.join()
            cnt += 1
            print(cnt)
            
                # exp = utils.Run_Experiment(model        = model.math_model,
                #                            parameters   = parameters,
                #                            eval         = Auth.evaluate,
                #                            m_size       = 128,
                #                            t_size       = 256,
                #                            save         = True)


# parameters = {'m_nr': 25, 't_nr': 25, 
#               'p': 0.95, 'q': 1, 
#               'TagEveryMessage': True, 
#               'AtLeastOnce': False, 
#               'EquivalentA': True}

# exp = utils.Run_Experiment(model        = model.math_model,
#                            parameters   = parameters,
#                            eval         = Auth.evaluate,
#                            m_size       = 1024,
#                            t_size       = 256,
#                            save         = True)
# exp['eval']




### Looking at the saved Models

In [ ]:
exp = utils.Load_Experiments()

import seaborn as sns

for i in range(len(exp)):
    print(exp[i]['parameters'])
    print(exp[i]['eval'])
    sns.heatmap(exp[i]['results']['X'])
    plt.show()
    print('\n\n')

### Creating a  progressive MAC for comparison

In [ ]:
##### AGG MAC 3 ####
# X = np.array([  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
#                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
#                 ])

# parameters = {'m_nr': X.shape[0], 't_nr': X.shape[1],
#                 'p': 0.85, 'q': 1}

parameters = {'m_nr': 20, 't_nr': 20,
                'p': 0.85, 'q': 1,
                'x': 4}
X = Auth.ProMAC_X(parameters['m_nr'],parameters['x'])


exp = Auth.Create_Experiment(parameters,X= X)

exp['eval'] = Auth.evaluate(exp,m_size=1024,t_size=int(256/4), b = 256 )
utils.Save_Experiment(exp)
print(exp)


# exp = Auth.evaluate(exp,m_size=1024,t_size=int(256/4), b = 256 )

### Run the optimizer With Latency

In [2]:
parameters = {'m_nr': 10, 't_nr': 10,
                'p': 0.9, 'q': .9,
                'TagEveryMessage': True,
                'AtLeastOnce': False,
                'EquivalentA': True,
                'weight_A': 1,
                'weight_L': 0}


for i in range(10):
    exp = utils.Run_Experiment(model = model_latency.math_model,
                            parameters = parameters,
                            eval=Auth.evaluate,
                            save=True,
                            m_size=1024,
                            t_size=256)
    parameters['weight_L'] += 0.1
    print(exp['eval'])

# exp['eval']

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-04
Status: 1
Objective value: 1.0
Experiment saved as experiment number 0
{'A': array([3.13810596, 3.13810596, 3.13810596, 3.13810596, 3.13810596,
       3.13810596, 3.13810596, 3.13810596, 3.13810596, 3.13810596]), 'L': array([9, 8, 7, 6, 5, 4, 3, 2, 1, 0]), 'average_A': 3.1381059609000017, 'average_L': 4.5, 'computation_(tag to message ratio)': 1.0, 'goodput_without_tag_adjustment': 0.8, 'goodput_with_tag_adjustment': 0.9266968325791856, 'security_goodput': 1.0, 'rows_that_breaks_the_verification': [10]}
Status: 1
Objective value: 3.0
Experiment saved as experiment number 1
{'A': array([2.36196, 2.36196, 2.36196, 2.36196, 2.36196, 2.36196, 2.36196,
       2.36196, 2.36196, 2.36196]), 'L': array([3, 2, 1, 0, 0, 1, 0, 0, 1, 0]), 'average_A': 2.3619600000000007, 'average_L': 0.8, 'computation_(tag to message ratio)': 1.0, 'goodput_without_tag_adjustment': 0.8, 'goodput_with_tag_adjustment': 0.9045936